# The Transformer Block and the Residual Stream

Topic 9 read attention as a kernel smoother, and smoothing averages. Stack averaging operators and every token drifts
toward the mean: the cloud of token vectors collapses. The residual connection is the term that keeps each token
itself. With it, the block becomes a shared $d$-dimensional **stream** that every head reads with a linear map and
writes into by addition — and that additivity makes the logits an exact sum of paths, keeps an identity path open for
the gradient, and lets two heads compose into an **induction head**.

This notebook imports the topic's module, which owns every number; each cell calls a function and prints what the page
quotes.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "transformer_block_residual_stream.py").exists()
                       else here / "notebooks" / "transformer-block-residual-stream"))
import transformer_block_residual_stream as tb

S = tb.summary()
print("collapse at beta = 1, 12 layers:  pure diam", tb.collapse_run(1.0, False)["diam"][-1],
      "| with skip, diam grows", round(tb.collapse_run(1.0, True)["diam"][-1] / tb.collapse_run(1.0, True)["diam"][0]), "x")
print("names never seen in training:     induction model", round(S["names_unseen_induction-2"], 3),
      "| filing-trained model", round(S["names_unseen_filings-2"], 3))

## 1. The block, assembled

**Def 1.** A head $h$ sends $X \in \mathbb{R}^{n\times d}$ to
$\operatorname{softmax}(XW_Q^h(XW_K^h)^\top/\sqrt{d_h} + B)\,XW_V^hW_O^h$; multi-head attention is the sum over heads; an
MLP is $\sigma(xW_1 + b_1)W_2 + b_2$ on each row. The **pre-LN block** is $X \leftarrow X + \mathrm{MHA}(\mathrm{LN}\,X)$,
then $X \leftarrow X + \mathrm{MLP}(\mathrm{LN}\,X)$; the **post-LN block** puts LayerNorm after each addition.

**Prop 1.** (i) Multi-head attention is $\sum_h \mathrm{Attn}(XW_Q^h, XW_K^h, XW_V^h)\,W_O^h$ — Topic 9's smoother $H$
times, each copy mapped into the stream by its own $W_O^h$. (ii) The MLP is a key–value memory,
$b_2 + \sum_i \sigma(\langle x, k_i\rangle + b_{1i})\,v_i$, with keys the columns of $W_1$ and values the rows of $W_2$.

In [ ]:
tb.test_multi_head_is_a_sum_of_smoothers()
print("Prop 1 (i) against Topic 9's attention() and (ii) against the key-value sum: both to 1e-12")

## 2. Attention alone collapses

**Def 2.** For a row-stochastic $A$, the Dobrushin coefficient is
$\tau(A) = \tfrac12\max_{i,j}\lVert A_{i\cdot} - A_{j\cdot}\rVert_1 = 1 - \min_{i,j}\sum_k \min(A_{ik}, A_{jk})$.

**Thm 1.** (i) $\operatorname{diam}(AX) \le \tau(A)\operatorname{diam}(X)$ in every norm. (ii) If each row of the scores
has range $\le r$, then $\tau(\operatorname{softmax} S) \le 1 - e^{-r}$. (iii) The pure stack $X_{l+1} = A_lX_lW_l$ has
$\operatorname{diam}(X_L) \le \prod_l \tau(A_l)\lVert W_l\rVert\,\operatorname{diam}(X_0)$.

*Proof sketch of (i).* Rows $i, j$ of $AX$ differ by $\sum_k c_kx_k$ with $c = A_{i\cdot} - A_{j\cdot}$ summing to zero;
split $c = \tau_{ij}(p - q)$ into two probability vectors, and the difference is
$\tau_{ij}\sum_{k,m}p_kq_m(x_k - x_m)$, of norm at most $\tau(A)\operatorname{diam}(X)$.

In [ ]:
tb.test_dobrushin_contraction()
rng = np.random.default_rng(0)
A = rng.dirichlet(np.ones(6), size=6)
X = rng.normal(size=(6, 2))
print(f"tau(A) = {tb.dobrushin(A):.4f};  diam(AX)/diam(X) = {tb.diameter(A @ X) / tb.diameter(X):.4f}")

The lab's cloud: ten tokens in the plane, $A_l = \operatorname{softmax}(\beta X_lX_l^\top)$, $W = I$. A wide kernel
(small $\beta$, Topic 9's high temperature) has $\tau$ well below 1 and the pure stack collapses; a narrow kernel makes
each token attend to itself, $\tau \approx 1$, and the bound is silent — the cloud freezes into clusters.

**Prop 2.** With the skip, $\operatorname{diam}(X + AXW) \ge (1 - \tau(A)\lVert W\rVert)\operatorname{diam}(X)$: the skip
adds the differences back unchanged.

In [ ]:
print(" beta | tau_0  | pure diam_12 | pure spread_12 | skip diam_12/diam_0 | skip spread_12")
for b in tb.COLLAPSE_BETAS:
    p, s = tb.collapse_run(b, False), tb.collapse_run(b, True)
    print(f"{b:5} | {p['tau'][0]:.3f}  | {p['diam'][-1]:12.3g} | {p['spread'][-1]:14.4f} | "
          f"{s['diam'][-1] / s['diam'][0]:19.1f} | {s['spread'][-1]:.4f}")
tb.test_collapse_and_the_skip()

Read the last two columns together at $\beta = 0.25$: the diameter grows, yet the relative spread
$\lVert P_nX\rVert_F/\lVert X\rVert_F$ falls to 0.0016. The skip keeps differences, not proportions — with
$A \approx \mathbf{1}\mathbf{1}^\top/n$, $X + AX$ adds the mean row to every token, and the common part doubles each
layer.

## 3. LayerNorm is a sphere in a hyperplane

**Prop 3.** $\mathrm{LN}(x) = \sqrt d\,Px/\lVert Px\rVert$ with $P = I - \mathbf{1}\mathbf{1}^\top/d$: project onto
$\mathbf{1}^\perp$, then radially onto the sphere of radius $\sqrt d$ in it. It ignores scale and shifts along
$\mathbf{1}$, and is idempotent.

**Cor 1.** $\partial\mathrm{LN}/\partial x = (\sqrt d/\lVert Px\rVert)(P - uu^\top)$, $u = Px/\lVert Px\rVert$: symmetric,
it kills $\mathbf 1$ and $x$, and its singular values are $\sqrt d/\lVert Px\rVert$ ($d - 2$ times) and 0 (twice).

In [ ]:
tb.test_layer_norm_geometry()
tb.test_thin_shell()
x = np.array([3.0, 1.0, -0.5])
J = tb.ln_jacobian(x)
print("LN(x) =", np.round(tb.layer_norm(x), 4), " |LN(x)| =", round(float(np.linalg.norm(tb.layer_norm(x))), 4), "= sqrt 3")
print("J @ 1 =", np.round(J @ np.ones(3), 12), " J @ x =", np.round(J @ x, 12),
      " singular values:", np.round(np.linalg.svd(J, compute_uv=False), 4))

## 4. The identity path

**Thm 2.** (i) For $x_{k+1} = x_k + f_k(x_k)$, $\partial x_L/\partial x_l = \prod_k (I + J_k)$, which multiplies out to a
sum over the $2^{L-l}$ subsets of layers — and the empty subset is $I$, so every layer receives the top gradient
unchanged. (ii) Pre-LN: $J_k = J_{F_k}J_{\mathrm{LN}}(x_k)$ with $\lVert J_{\mathrm{LN}}\rVert = \sqrt d/\lVert Px_k\rVert$,
so the branches shrink as the stream grows while $I$ does not. (iii) Post-LN: every path passes through $L - l$
LayerNorm Jacobians.

In [ ]:
tb.test_jacobian_paths()
print("Thm 2 (i): the 16 subset terms equal the product, and the product equals the stack's Jacobian")

Xiong et al. (their Theorem 1) bound the last FFN layer's gradient at initialization by $O(d\sqrt{\ln d})$ for post-LN
and $O(d\sqrt{\ln d / L})$ for pre-LN. We do not prove it; we measure the scaling it predicts on random stacks
($d = 64$, four heads, a ReLU MLP of width 256; 8 draws per depth).

In [ ]:
print("  L | pre-LN last dW2 | post-LN last dW2 | pre-LN |x_L| | post-LN |x_L|")
for L in tb.GRAD_DEPTHS:
    pre, post = tb.gradient_profile("pre", L), tb.gradient_profile("post", L)
    print(f"{L:3} | {pre['w2'][-1]:15.6f} | {post['w2'][-1]:16.6f} | {pre['norm'][-1]:12.6f} | {post['norm'][-1]:.6f}")
tb.test_gradient_scale_at_init()

## 5. The stream as a channel

**Def 4.** The attention-only model: $x_0 = E[t]$, $x_{l+1} = x_l + \mathrm{MHA}_l(x_l)$ (causal, RoPE), logits
$x_LU$ — no MLP, no LayerNorm. Head $(l, h)$ writes through its **OV circuit** $W_{OV}^{lh} = W_V^{lh}W_O^{lh}$.

**Thm 3.** Given each head's attention matrix from the forward pass, the logits are the sum over chains of heads
$(l_1,h_1),\dots,(l_m,h_m)$, $l_1 < \dots < l_m$, of
$A^{l_mh_m}\cdots A^{l_1h_1}\,x_0\,W_{OV}^{l_1h_1}\cdots W_{OV}^{l_mh_m}\,U$. The empty chain is the direct path $E[t]U$.

The model below is the trained 2-layer induction model of §6; with two heads per layer it has $1 + 4 + 4 = 9$ paths.

In [ ]:
P2 = tb.train_model(tb.SYNTH2)
tok = tb.repeated_test_set()[0][:50]
paths = tb.path_logits(P2, tok)
err = np.abs(sum(paths.values()) - tb.model_forward(P2, tok)[1]).max()
print(len(paths), "paths:", [("+".join(f"{l}.{h}" for l, h in c) or "direct") for c in paths])
print(f"max |sum of paths - forward logits| = {err:.2e}")
tb.test_path_expansion()

**Prop 4.** If every layer-0 head writes into a subspace the layer-1 keys do not read
($W_{OV}^{0h}W_K^{1h'} = 0$), layer 1's attention is the same with or without layer 0's output: heads talk only through
the subspaces where one writes and another reads.

In [ ]:
tb.test_orthogonal_heads_do_not_talk()
print("Prop 4: K-composition score 0, and layer 1's patterns unchanged to 1e-12")

## 6. Induction heads

The task: a random block of $V = 256$ tokens with period $p \sim U\{6,\dots,24\}$, repeated to 48 tokens. From
position $p$ on, the next token is predictable — by looking up where the current token occurred before and copying
what came *after* it: `[A][B] … [A] → [B]`. Two models, 1000 Adam steps each, hand-written gradients.

In [ ]:
rows = (("2-layer", tb.repeated_metrics(P2)), ("1-layer", tb.repeated_metrics(tb.train_model(tb.SYNTH1))),
        ("2-layer, keys from x0", tb.repeated_metrics(P2, ablate_k=True)))
for name, m in rows:
    print(f"{name:22} loss {m['loss']:.6f} nats   accuracy {m['acc']:.6f}")
print(f"{'uniform':22} loss {np.log(256):.6f}")

The heads: one layer-0 head attends to the previous token — its content-free RoPE score (Topic 10's kernel over
offsets, built from its biases) peaks at offset 1 — and both layer-1 heads attend to the position after the current
token's previous occurrence.

In [ ]:
hs = tb.head_scores(P2)
h = int(hs["prev"][0].argmax())
print("previous-token weight, layer 0 heads:", np.round(hs["prev"][0], 6))
print("induction weight,      layer 1 heads:", np.round(hs["induction"][1], 6))
print(f"head (0,{h}) content-free score by offset 0..5:", np.round(tb.bias_offset_score(P2, 0, h, range(6)), 3))
for k in "kqv":
    print(f"{k.upper()}-composition scores:", np.round(tb.composition_scores(P2, k).ravel(), 3))

**Prop 5.** In one layer, the score $s_h(i, j)$ is a function of $(x_i, i, x_j, j)$ alone — it cannot see token
$j - 1$, which is exactly what induction must check. In two layers, the previous-token head writes $x_{j-1}$ into the
stream at $j$, and layer 1's keys read it (K-composition).

In [ ]:
tb.test_one_layer_keys()
print("Prop 5: changing token j-1 leaves the 1-layer score unchanged; it moves the 2-layer score by > 2 nats")

In [ ]:
tb.test_composed_path_carries_the_copy()
print("the paths ending in a layer-1 head carry > 90% of the correct token's centered logit (median)")

**The logit lens.** Read the stream through $U$ after each layer. This model has no final LayerNorm, so the lens is
its own readout; on models with one, $W_U\,\mathrm{LN}(x_l)$ is a heuristic.

In [ ]:
lens = tb.logit_lens(P2)
print("P(correct token) after layer 0, 1, 2:", np.round(lens, 6), "  chance:", round(1 / 256, 6))

## 7. Finance case study: completing a company's name

`split_roster()` holds out a quarter of the public companies. We train on filings about the rest, then ask every
reader to complete multi-word company names that were already mentioned earlier in the same filing — on new filings
about the *seen* companies, and on filings about companies whose names never occur in training.

In [ ]:
acc = tb.name_accuracy()
print(f"{'reader':16} {'seen':>8} {'unseen':>8}")
for k in tb.READERS:
    print(f"{k:16} {acc['seen'][k]:8.6f} {acc['unseen'][k]:8.6f}")
print("queries:", {s: len(tb.name_queries(s)['Y']) for s in ('seen', 'unseen')})
tb.test_finance_names()

Memory in the weights is a lookup table: the bigram table and the filing-trained model complete names they were
trained on and none of the others. The induction model never saw a filing; it completes the unseen names because it
reads them off the filing's own earlier text. The filing-trained model, same architecture, never learned to copy —
in training, the names were never new.

In [ ]:
fi = tb.lab_filing()
doc = tb.roster_docs("unseen")[fi]
Q = tb.name_queries("unseen")
pred = tb.readers_predict(Q["W"])
for q, (d, i) in enumerate(Q["where"]):
    if d == fi:
        hits = "".join("x" if pred[k][q] == Q["Y"][q] else "." for k in tb.READERS)
        print(f"{' '.join(doc.words[i - 2:i + 1]):40} -> {doc.words[i + 1]:10} {hits}  ({', '.join(tb.READERS)})")

## The lab

`ResidualStreamLab` reads `viz_payload()`, written to `src/components/viz/data/residual-stream-lab.json` and pinned by
`test_viz_mirror_matches_payload`.

In [ ]:
p = tb.viz_payload()
print({k: list(v)[:6] for k, v in p.items()})
tb.test_viz_payload()

In [ ]:
assert tb._run_harness() == 0